In [2]:
import torch

inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],  # Your (x^1)
        [0.55, 0.87, 0.66],  # journey (x^2)
        [0.57, 0.85, 0.64],  # starts (x^3)
        [0.22, 0.58, 0.33],  # with (x^4)
        [0.77, 0.25, 0.10],  # one (x^5)
        [0.05, 0.80, 0.55],  # step (x^6)]
    ]  
)

/Users/benoit/Projects/build-a-llm-exercices/.venv/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:279: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [3]:
inputs.size() #(6 rows, 3 colunns )

torch.Size([6, 3])

In [4]:
query = inputs[1] # journey
attn_scores_2 = torch.empty(inputs.shape[0]) 
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)
print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [5]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
attn_weights_2_tmp

tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])

In [6]:
def softmax_naive(x):
    return torch.exp(x)/torch.exp(x).sum(dim=0)

In [7]:

attn_weights_2_naive = softmax_naive(attn_scores_2)
attn_weights_2_naive

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [8]:
# PyTorch better

attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
attn_weights_2

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [9]:
query = inputs[1]
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


In [10]:
query = inputs[1] # journey
attn_scores_2 = torch.empty(inputs.shape[0]) # 6
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)
    
attn_weights_2_naive = softmax_naive(attn_scores_2)

context_vec_2 = torch.zeros(query.shape) # 3
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


But we only computed for one input!


In [11]:
attn_scores = inputs @ inputs.T
attn_scores

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

In [12]:
# apply softmax by the last dimension, so by columns (row,columns)
attn_weights = torch.softmax(attn_scores, dim=-1) 
attn_weights

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [13]:
attn_weights.sum(dim=-1) # sum on columns is indeed 1

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])

In [14]:
all_context_vecs = attn_weights @ inputs
all_context_vecs

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

#### Summary


In [15]:
attn_scores = inputs @ inputs.T
attn_weights = torch.softmax(attn_scores, dim=-1) 
all_context_vecs = attn_weights @ inputs
all_context_vecs

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

#### 3.4


In [16]:
x_2 = inputs[1] # Our input
d_in = inputs.shape[1] # Input dimension - 3
d_out = 2  #Output dimension - 2

In [17]:
torch.manual_seed(123)

Think of requires_grad=True as saying "I want to be able to compute how changes to this tensor affect the final loss" — which is exactly what you need to train a neural network.

In [18]:
W_query =  torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key =  torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value =  torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [19]:
query_2 = x_2 @ W_query 
key_2 = x_2 @ W_key 
value_2 = x_2 @ W_value

In [20]:
query_2

tensor([0.4306, 1.4551])

but we need to compute for everything, regardless of query

In [21]:
keys = inputs @ W_key
values = inputs @ W_value


In [22]:
keys.shape

torch.Size([6, 2])

In [23]:
values.shape

torch.Size([6, 2])

In [24]:
keys_2 = keys[1]
attn_scores_22 = query_2.dot(keys_2)
print(attn_scores_22)

tensor(1.8524)


In [25]:
attn_scores_2 = query_2 @ keys.T

In [26]:
attn_scores_2

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])

In [27]:
d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2/d_k**0.5, dim = -1)
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


In [28]:
context_vec_2 = attn_weights_2 @ values
context_vec_2

tensor([0.3061, 0.8210])

In [ ]:
from torch import Tensor
import torch.nn as nn

class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.d_out = d_out
        self.W_query =  torch.nn.Parameter(torch.rand(d_in, d_out))
        self.W_key  =  torch.nn.Parameter(torch.rand(d_in, d_out))
        self.W_value =  torch.nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, input: Tensor):
        queries = input @ self.W_query
        keys = input @ self.W_key
        values = input @ self.W_value
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores/self.d_out**0.5, dim = -1)
        context_vec = attn_weights @ values
        return context_vec




In [30]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in,d_out)

In [31]:
sa_v1((inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)

In [34]:

class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.d_out = d_out 
        self.W_query =  nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key  =  nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value =  nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores/keys.shape[-1]**0.5, dim=-1)
        context_vec = attn_weights @ values
        return context_vec
    




In [35]:
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)

In [37]:
sa_v2(inputs)

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)

In [60]:
sa_v1.W_key = torch.nn.Parameter(sa_v2.W_key.weight.T)
sa_v1.W_query = torch.nn.Parameter(sa_v2.W_query.weight.T)
sa_v1.W_value = torch.nn.Parameter(sa_v2.W_value.weight.T)

In [61]:
sa_v1(inputs)

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)